# Sahformer — Colab training

Clock-aware Chessformer (faithful Maia-3 5M backbone + our time layer) on a GPU runtime.

**First:** Runtime → Change runtime type → **GPU (T4)**.

Then edit `REPO_URL` in the next cell and run top to bottom. Checkpoints stream to Google Drive.

In [ ]:
# 1) Get the code
REPO_URL = "https://github.com/YOUR_USER/YOUR_REPO.git"  # <-- EDIT ME
import os
repo_dir = REPO_URL.rstrip("/").split("/")[-1].replace(".git", "")
if not os.path.isdir(repo_dir):
    !git clone "$REPO_URL"
%cd $repo_dir
!git pull --ff-only || true

In [ ]:
# 2) Deps + GPU check (torch ships with Colab)
!pip -q install python-chess zstandard
import sys; sys.path.insert(0, ".")
import torch
ok = torch.cuda.is_available()
print("torch", torch.__version__, "| cuda", ok, "|",
      torch.cuda.get_device_name(0) if ok else "NO GPU -> Runtime > Change runtime type > GPU")

## 3) Build training shards (streamed from Lichess, chunked)

Streams a 2017-04+ month (these have `%clk` clocks) and early-stops, so only the first
part of the archive transfers. It writes the data in **chunks** (`build_shards`), so memory
stays flat no matter how big `MAX_POSITIONS` is — you can go large here. Notes:
- Output is several `data/shard0000.npz`, `shard0001.npz`, … files; training reads them all.
- `BALANCE=True` equalizes Elo bins per chunk; the rare extreme-Elo bins shrink the set a lot.
  For a first real run we default `BALANCE=False` (more volume, Elo-skewed).

In [ ]:
# 3) Fetch + build (chunked, memory stays flat)
URL = "https://database.lichess.org/standard/lichess_db_standard_rated_2017-04.pgn.zst"
MAX_POSITIONS = 600000   # go big — memory is flat because it writes in chunks
BALANCE = False

import glob
from sahformer.download import stream_games_from_url
from sahformer.dataset_build import build_shards, records_from_games

build_shards(
    records_from_games(stream_games_from_url(URL)),
    "data", chunk_positions=150000, max_positions=MAX_POSITIONS,
    balance=BALANCE, progress_every=5000)
DATA_SHARDS = sorted(glob.glob("data/*.npz"))
print("shards:", DATA_SHARDS)

## 4) Mount Drive for checkpoints

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/sahformer_ckpts"
import os; os.makedirs(OUT, exist_ok=True)
print("checkpoints ->", OUT)

## 5) Train the clock-aware model (`full`) on GPU with AMP

Trains to fast local disk, then copies `best.pt` / `last.pt` to Drive at the end (Drive writes
mid-run are slow). Bump `max_steps` for a longer run; watch the loss curve.

In [ ]:
import glob, os, shutil
from sahformer.training.loop import TrainConfig, train
DATA_SHARDS = sorted(glob.glob("data/*.npz"))

LOCAL = "/content/ckpts_full"     # train to fast local disk, copy to Drive at the end
cfg = TrainConfig(mode="full", max_steps=12000, warmup_steps=600, batch_size=512,
                  lr=3e-4, amp=True, device="cuda", out_dir=LOCAL,
                  log_every=200, ckpt_every=2000)
res = train(cfg, DATA_SHARDS)

os.makedirs(f"{OUT}/full", exist_ok=True)
for f in ("best.pt", "last.pt"):
    shutil.copy(f"{LOCAL}/{f}", f"{OUT}/full/{f}")
print("full best_total:", res["best"], "| saved to", f"{OUT}/full")

In [ ]:
import matplotlib.pyplot as plt
h = res["history"]; xs = [r["step"] for r in h]
for key in ("total", "policy", "time"):
    plt.plot(xs, [r[key] for r in h], label=key)
plt.legend(); plt.xlabel("step"); plt.ylabel("loss"); plt.title("full — losses"); plt.show()
plt.plot(xs, [r["move_acc"] for r in h]); plt.xlabel("step"); plt.ylabel("move_acc")
plt.title("full — move accuracy (sanity metric)"); plt.show()

## 6) Optional: baseline (clock-blind) for the later ablation comparison

In [ ]:
LOCAL_B = "/content/ckpts_baseline"
cfg_b = TrainConfig(mode="baseline", max_steps=12000, warmup_steps=600, batch_size=512,
                    lr=3e-4, amp=True, device="cuda", out_dir=LOCAL_B,
                    log_every=200, ckpt_every=2000)
res_b = train(cfg_b, DATA_SHARDS)
os.makedirs(f"{OUT}/baseline", exist_ok=True)
for f in ("best.pt", "last.pt"):
    shutil.copy(f"{LOCAL_B}/{f}", f"{OUT}/baseline/{f}")
print("baseline best:", res_b["best"], "| full best:", res["best"])

## Done

Checkpoints are on your Drive under `sahformer_ckpts/`. **Don't over-read baseline-vs-full
here** — a rigorous comparison (move-match vs Maia by Elo, think-time calibration, sampled
non-deterministic play) is the next plan. This run just trains the model for real on GPU.